[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonardijulia/AI4Good/blob/main/HandsOn_Part2/TerraTorch_HandsOn_Part2_Template.ipynb)

## ‼️Before running any code ‼️
### **Runtime ➡️ Change runtime type ➡️ choose a runtime with GPU**

# 🛠️ Setting up the environment
- Install terratorch (restart the environment as prompted)
- Import the necessary python libraries


In [ ]:
!pip install terratorch

In [ ]:
import os
import torch
import numpy as np

from huggingface_hub import hf_hub_download

import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

from lightning.pytorch import Trainer
from terratorch.tasks import ClassificationTask
from terratorch.datamodules import MEuroSATNonGeoDataModule

# 📊 Import the dataset
## EuroSAT Image Classification dataset

- Satellite (S2L1C) LULC dataset;
- https://github.com/phelber/EuroSAT
- Multispectral (13 bands);
- 10 LULC classes *'Annual Crop', 'Forest, 'Herbaceus Vegetation', 'Highways', 'Industrial Buildings', 'Pasture', 'Permanent Crop', 'Residential Bildings', 'River', 'Sea & Lake'*;

In [ ]:
# Load the dataset
hf_hub_download(
    repo_id="recursix/geo-bench-1.0",
    filename="classification_v1.0/m-eurosat.zip",
    repo_type="dataset",
    local_dir="/content/data/EuroSAT"
    )

### 🧹 Clean the dataset
Remap the classes in a consistent way and save them in the corresponding folders

In [ ]:
!unzip -q /content/data/EuroSAT/classification_v1.0/m-eurosat.zip -d /content/data/EuroSAT/

In [ ]:
!wget -P /content/data/EuroSAT/m-eurosat/ https://raw.githubusercontent.com/leonardijulia/AI4Good/refs/heads/main/HandsOn_Part2/label_map.json

### 🗺️ Visualize an example from the dataset

In [ ]:
import h5py
import json
example = 'id_16500'
label_map = json.load(open("/content/data/EuroSAT/m-eurosat/label_map.json"))
label = [k for k,v in label_map.items() if example in v]

with h5py.File(f'/content/data/EuroSAT/m-eurosat/{example}.hdf5', 'r') as f:
  keys = sorted(f.keys())
  keys = np.array([key for key in keys if key != "label"])
  bands = [np.array(f[key]) for key in keys]

  image = np.stack(bands, axis=-1)
image = image.astype(np.float32)
image_vis = image[:,:,[3,2,1]]
image_vis = (image_vis - image_vis.min(axis=(0,1))) * (1 / image_vis.max(axis=(0,1)))
image_vis = np.clip(image_vis, 0, 1)

In [ ]:
plt.imshow(image_vis)
plt.title(f"Label: {label[0]}")
plt.show()

# 📑 Set up the datamodule

TerraTorch offers the datamodule for EuroSAT.

In [ ]:
###

# ⚙️ Set up the model

In [ ]:
from terratorch.registry import TERRATORCH_BACKBONE_REGISTRY

# browse for backbones - what GFM do we want to use?

In [ ]:
# Set up the Backbone, Neck, Decoder, and Head parameters:

model_args = dict(
    # Backbone setup:

    # Decoder setup:

    # Neck setup:

    # Head setup:
)

In [ ]:
# Define the TerraTorch task using the model arguments:


In [ ]:
# Define the Lightning trainer:


# 🧠 Fine-tune the model

In [ ]:
# With everything set up, the training is run calling this one line of code:


# ✅ Test the fine-tuned model

In [ ]:
# Load the best checkpoint
# Run the test run using the best ckpt

## 🖼️ Visualize the predictions

In [ ]:
model = ClassificationTask.load_from_checkpoint(
    #best_ckpt_path,
    #model_factory=task.hparams.model_factory,
    #model_args=task.hparams.model_args,
).cuda()

#test_loader = datamodule.test_dataloader()

with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch["image"].to(model.device)
    masks = batch["label"].numpy()

    with torch.no_grad():
        outputs = model(images)

    preds = torch.argmax(outputs.output, dim=1).cpu().numpy()

for i in range(8):
    sample = {
        "image": batch["image"][i].cpu(),
        "label": batch["label"][i],
        "prediction": preds[i],
    }
    image_vis = sample["image"].permute(1, 2, 0).numpy()
    img = image_vis[:, :, [3, 2, 1]]
    img = (img - img.min(axis=(0, 1))) * (1 / img.max(axis=(0, 1)))
    img = np.clip(img, 0, 1)
    plt.imshow(img) # Changed from plt.plot to plt.imshow
    plt.title(f"Label: {sample['label']}, Prediction: {sample['prediction']}")
    plt.show()

# 📜 Using the YAML file - excercise
- Create the .yaml file based on this notebook
- Run it from the 'CLI' interface
- Experiment with seetings (backbones, decoders, necks, transforms,...)